# Lezione 8 · Pratica: Attenzione e Transformer

## Cosa farai in questa pratica

In questo notebook applichi quanto visto nella Lezione 8 su attenzione e Transformer. Completi il codice in due esercizi:

- **Esercizio 1.** Implementi da zero l'attenzione scalata con prodotto scalare (*scaled dot-product attention*), il blocco che confronta una query con un insieme di chiavi e combina i valori corrispondenti in base a quel confronto.
- **Esercizio 2.** Estendi quella funzione all'attenzione multi-testa (*multi-head attention*): dividi query, chiavi e valori in più teste, applichi l'attenzione dell'Esercizio 1 a ciascuna separatamente e concateni i risultati.

Ogni esercizio include lo scheletro del codice con parti contrassegnate come `# TODO`, che devi completare tu. Sotto ogni esercizio trovi una soluzione collassata: prova a risolvere l'esercizio da solo prima di aprirla.

## Prerequisito

Devi aver seguito la Lezione 8 ("Attenzione e Transformer"), in cui si spiega il collo di bottiglia delle reti neurali ricorrenti (RNN) su sequenze lunghe, il meccanismo di attenzione con query, chiave e valore (*query, key, value*), e la formula dell'attenzione scalata con prodotto scalare:

$$\text{Attenzione}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V$$

## Come eseguire questo notebook

Aprilo in Google Colab ed esegui le celle in ordine, dall'alto verso il basso. Non installare nulla di aggiuntivo: `torch` è già preinstallato in Colab. Non serve attivare alcun acceleratore: i tensori di questo notebook sono piccoli e girano rapidamente sul processore standard di Colab.

## Esercizio 1 · Attenzione scalata con prodotto scalare da zero

### Il caso

Fai parte del team tecnico di un hub di innovazione digitale che prepara un laboratorio interno sui Transformer per gli enti partner. Prima di insegnare a chiunque a usare `nn.MultiheadAttention`, vuoi che il team capisca cosa succede dentro quella scatola nera. Ti occupi di programmare, con le operazioni di base di PyTorch, esattamente la formula vista nella Lezione 8: confronti una query con un insieme di chiavi, trasformi quel confronto in pesi con softmax e usi quei pesi per combinare i valori.

### Cosa devi fare

Completa il corpo della funzione `atencion_escalada(Q, K, V)`. Riceve tre matrici — query, chiavi e valori — e restituisce l'output dell'attenzione insieme ai pesi di attenzione utilizzati. Il resto dello scheletro (i dati di esempio, la funzione `softmax_manual` e la cella che prova la funzione) è già scritto: non modificarlo.

### Suggerimenti

- La dimensione delle chiavi, $d_k$, è la dimensione dell'ultimo asse di `K`: ottienila con `K.size(-1)`.
- Moltiplica `Q` per la trasposta di `K` sui suoi due ultimi assi con `K.transpose(-2, -1)`, non con `K.T` (così la funzione funziona anche se in seguito aggiungi una dimensione di lotto).
- Scala il risultato dividendo per la radice quadrata di $d_k$, come visto in teoria: `d_k ** 0.5`.
- Applica `softmax_manual` sull'ultimo asse (`dim=-1`), in modo che ogni riga di pesi sia una distribuzione di probabilità che somma a uno.
- L'output finale è il prodotto dei pesi per `V`.

In [ ]:
import torch

torch.manual_seed(0)

# Frase de ejemplo, ya tokenizada en palabras
tokens = ["el", "hub", "conecta", "empresas", "con", "tecnologia", "avanzada"]
n = len(tokens)
d = 8    # dimensión del embedding, pequeña para que el ejemplo sea legible

# Embeddings de partida: en un modelo entrenado vendrían de una capa de
# embedding aprendida; aquí los generamos al azar para centrarnos en la
# mecánica de la atención
X = torch.randn(n, d)

# Proyecciones lineales aprendibles: transforman el mismo embedding en
# tres papeles distintos (consulta, clave, valor)
W_q = torch.randn(d, d) * 0.5
W_k = torch.randn(d, d) * 0.5
W_v = torch.randn(d, d) * 0.5

Q = X @ W_q
K = X @ W_k
V = X @ W_v


def softmax_manual(z, dim=-1):
    # Restamos el máximo antes de la exponencial por estabilidad numérica
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion_escalada(Q, K, V):
    """Calcula la atención escalada por producto punto.

    Devuelve una tupla (salida, pesos):
    - salida: combinación ponderada de V, misma forma que Q
    - pesos: pesos de atención, forma (n_consultas, n_claves)
    """
    d_k = K.size(-1)

    # TODO: calcula las puntuaciones de relevancia entre Q y K con producto
    # punto. Usa K.transpose(-2, -1) para trasponer las dos últimas
    # dimensiones de K antes de multiplicar
    puntuaciones = ...

    # TODO: escala las puntuaciones dividiendo entre la raíz cuadrada de d_k
    puntuaciones = ...

    # TODO: convierte las puntuaciones en pesos de atención con
    # softmax_manual, normalizando sobre la última dimensión
    pesos = ...

    # TODO: calcula la salida como combinación ponderada de V
    salida = ...

    return salida, pesos

In [ ]:
salida, pesos_atencion = atencion_escalada(Q, K, V)

print("Forma de la salida:", salida.shape)                     # (7, 8)
print("Cada fila de pesos suma:", pesos_atencion.sum(dim=-1))   # ~1.0 en cada posición

<details>
<summary>💡 Vedi soluzione</summary>

```python
def atencion_escalada(Q, K, V):
    """Calcola l'attenzione scalata con prodotto scalare.

    Restituisce una tupla (output, pesi):
    - output: combinazione ponderata di V, stessa forma di Q
    - pesi: pesi di attenzione, forma (n_query, n_chiavi)
    """
    d_k = K.size(-1)

    # Prodotto scalare tra ogni query e ogni chiave
    puntuaciones = Q @ K.transpose(-2, -1)

    # Scalatura per la radice di d_k, affinché il softmax non saturi
    puntuaciones = puntuaciones / (d_k ** 0.5)

    # Ogni riga diventa una distribuzione di probabilità
    pesos = softmax_manual(puntuaciones, dim=-1)

    # Combinazione ponderata dei valori
    salida = pesos @ V

    return salida, pesos
```
</details>

## Esercizio 2 · Attenzione multi-testa

### Il caso

La funzione che hai appena programmato funziona, ma costringe il modello a mescolare in un unico schema di pesi ogni tipo di relazione tra parole: vicinanza, concordanza grammaticale, riferimenti... Per lo stesso laboratorio interno, il team ti chiede ora il secondo blocco visto nella Lezione 8: l'attenzione multi-testa. Invece di un'unica attenzione, ne esegui diverse in parallelo — ciascuna su una porzione diversa della dimensione dell'embedding — e concateni le loro uscite.

### Cosa devi fare

Completa la funzione `atencion_multi_cabeza(Q, K, V, n_cabezas)`. Riutilizza `atencion_escalada`, la funzione che hai implementato nell'Esercizio 1. Lo scheletro che divide `Q`, `K` e `V` in teste, e quello che concatena le uscite alla fine, sono già scritti: il tuo compito è applicare `atencion_escalada` a ciascuna testa dentro il ciclo.

### Suggerimenti

- `Q`, `K` e `V` arrivano con forma `(n, d_modelo)`. Dopo la divisione già scritta, `Q_cabezas`, `K_cabezas` e `V_cabezas` hanno forma `(n_cabezas, n, d_cabeza)`, con `d_cabeza = d_modelo / n_cabezas`.
- Dentro il ciclo, `Q_cabezas[i]`, `K_cabezas[i]` e `V_cabezas[i]` sono le query, le chiavi e i valori della i-esima testa, con forma `(n, d_cabeza)`: esattamente ciò che si aspetta `atencion_escalada`.
- Chiama `atencion_escalada(Q_cabezas[i], K_cabezas[i], V_cabezas[i])` e salva l'uscita nella lista `salidas_cabezas`, e i pesi in `pesos_cabezas`.
- Non ti serve la matrice di proiezione finale $W^O$ della teoria: per questo esercizio basta concatenare le teste, come è scritto sotto il ciclo.

In [ ]:
def atencion_multi_cabeza(Q, K, V, n_cabezas):
    """Aplica atencion_escalada en paralelo sobre varias cabezas y
    concatena los resultados.

    Q, K, V: tensores de forma (n, d_modelo)
    n_cabezas: número de cabezas en las que dividir d_modelo

    Devuelve una tupla (salida, pesos_cabezas):
    - salida: forma (n, d_modelo), igual que la entrada
    - pesos_cabezas: lista con los pesos de atención de cada cabeza
    """
    n, d_modelo = Q.shape
    assert d_modelo % n_cabezas == 0, "d_modelo debe ser divisible entre n_cabezas"
    d_cabeza = d_modelo // n_cabezas

    # Dividimos la última dimensión en n_cabezas trozos y movemos el eje
    # de cabeza al frente: (n, d_modelo) -> (n_cabezas, n, d_cabeza)
    Q_cabezas = Q.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    K_cabezas = K.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    V_cabezas = V.view(n, n_cabezas, d_cabeza).transpose(0, 1)

    salidas_cabezas = []
    pesos_cabezas = []
    for i in range(n_cabezas):
        # TODO: aplica atencion_escalada a la cabeza i-ésima
        # (Q_cabezas[i], K_cabezas[i], V_cabezas[i]) y añade la salida a
        # salidas_cabezas y los pesos a pesos_cabezas
        ...

    # Apilamos las cabezas y volvemos a la forma original (n, d_modelo)
    salida = torch.stack(salidas_cabezas, dim=0).transpose(0, 1).reshape(n, d_modelo)

    return salida, pesos_cabezas

In [ ]:
n_cabezas = 2
salida_mc, pesos_mc = atencion_multi_cabeza(Q, K, V, n_cabezas)

print("Forma de la salida:", salida_mc.shape)                   # (7, 8), igual que Q
print("Número de cabezas devueltas:", len(pesos_mc))             # 2
print("Forma de los pesos de una cabeza:", pesos_mc[0].shape)    # (7, 4)

<details>
<summary>💡 Vedi soluzione</summary>

```python
def atencion_multi_cabeza(Q, K, V, n_cabezas):
    """Applica atencion_escalada in parallelo su più teste e
    concatena i risultati."""
    n, d_modelo = Q.shape
    assert d_modelo % n_cabezas == 0, "d_modelo deve essere divisibile per n_cabezas"
    d_cabeza = d_modelo // n_cabezas

    Q_cabezas = Q.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    K_cabezas = K.view(n, n_cabezas, d_cabeza).transpose(0, 1)
    V_cabezas = V.view(n, n_cabezas, d_cabeza).transpose(0, 1)

    salidas_cabezas = []
    pesos_cabezas = []
    for i in range(n_cabezas):
        salida_i, pesos_i = atencion_escalada(
            Q_cabezas[i], K_cabezas[i], V_cabezas[i]
        )
        salidas_cabezas.append(salida_i)
        pesos_cabezas.append(pesos_i)

    salida = torch.stack(salidas_cabezas, dim=0).transpose(0, 1).reshape(n, d_modelo)

    return salida, pesos_cabezas
```
</details>

## Per concludere

Hai appena costruito, con operazioni di base sui tensori, i due elementi che sostengono qualsiasi Transformer moderno: l'attenzione scalata con prodotto scalare e la sua versione multi-testa. Tutto il resto — la normalizzazione, la rete feed-forward, l'impilamento dei blocchi — si appoggia direttamente su ciò che hai appena programmato qui.

Prima della prossima lezione in diretta, prova a cambiare `n_cabezas` a quattro (usa allora un embedding di dimensione maggiore, per esempio 16, così la divisione resta esatta) e osserva come cambia la forma dei pesi di ciascuna testa.

**Condividi la tua soluzione e i tuoi dubbi nel canale del corso prima della prossima lezione in diretta!**